<a href="https://colab.research.google.com/github/mckore/DocsGPT/blob/main/CSE621_2026_Sentiment_Classification_of_Movie_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Classification of Movie Reviews
---
By: Kyle Spurlock

For: CSE-621

Last Updated: Feb 26, 2024





In [ ]:
import nltk.classify.util
from nltk.classify import NaiveBayesClassifier
import pandas as pd
import numpy as np

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

nltk.download('movie_reviews')
from nltk.corpus import movie_reviews # Load data from nltk data set: movie reviews

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


In [ ]:
# Function to convert sentences into feature vectors by considering every word as a feature.
def word_feats(words: list) -> dict:
  """
  Convert list of words to dictionary of (unique) words. This is the format used
  by NLTK to represent a document.

  Args:
    words (list): list of words/tokens occuring in document

  Returns:
    (dict): dictionary with words as keys (values just indicate the word occurs)
  """
  return dict([(word, True) for word in words])

# Extract positive and negative sentences
negids = movie_reviews.fileids('neg')
posids = movie_reviews.fileids('pos')
print('Positive review example: ' + ' '.join(movie_reviews.words(posids[0])))
print('Negative review example: ' + ' '.join(movie_reviews.words(negids[5])))

Positive review example: films adapted from comic books have had plenty of success , whether they ' re about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there ' s never really been a comic book like from hell before . for starters , it was created by alan moore ( and eddie campbell ) , who brought the medium to a whole new level in the mid ' 80s with a 12 - part series called the watchmen . to say moore and campbell thoroughly researched the subject of jack the ripper would be like saying michael jackson is starting to look a little odd . the book ( or " graphic novel , " if you will ) is over 500 pages long and includes nearly 30 more that consist of nothing but footnotes . in other words , don ' t dismiss this film because of its source . if you can get past the whole comic book thing , you might find another stumbling block in from hell ' s directors , albert and allen hughes . getting the hughes brothers t

NLTK expects the input to be represented in a sparse format, so each document d is represented by a tuple containing a dictionary with **w** keys, and the class label.

In [ ]:
# Convert sentences into feature vectors
negfeats = [(word_feats(movie_reviews.words(fileids=[f_id])), 'neg') for f_id in negids]
posfeats = [(word_feats(movie_reviews.words(fileids=[f_id])), 'pos') for f_id in posids]

# Create the dataset and split it into training and testing
negcutoff = int(len(negfeats) * 3/4)
poscutoff = int(len(posfeats) * 3/4)
trainfeats = negfeats[:negcutoff] + posfeats[:poscutoff]
testfeats = negfeats[negcutoff:] + posfeats[poscutoff:]
print('train on %d instances, test on %d instances' % (len(trainfeats), len(testfeats)))

# Train and evaluate the Naive Bayes classifier
classifier = NaiveBayesClassifier.train(trainfeats)
print('accuracy:', nltk.classify.util.accuracy(classifier, testfeats))
classifier.show_most_informative_features()

train on 1500 instances, test on 500 instances
accuracy: 0.728
Most Informative Features
             magnificent = True              pos : neg    =     15.0 : 1.0
             outstanding = True              pos : neg    =     13.6 : 1.0
               insulting = True              neg : pos    =     13.0 : 1.0
              vulnerable = True              pos : neg    =     12.3 : 1.0
               ludicrous = True              neg : pos    =     11.8 : 1.0
                  avoids = True              pos : neg    =     11.7 : 1.0
             uninvolving = True              neg : pos    =     11.7 : 1.0
              astounding = True              pos : neg    =     10.3 : 1.0
             fascination = True              pos : neg    =     10.3 : 1.0
                 idiotic = True              neg : pos    =      9.8 : 1.0


NLTK also allows you to easily wrap Sklearn classifiers and compute accuracy.

In [ ]:
from nltk.classify.scikitlearn import SklearnClassifier
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, GaussianNB

classifier = SklearnClassifier(BernoulliNB()).train(trainfeats)
print('Bernoulli NB accuracy:', nltk.classify.util.accuracy(classifier, testfeats))

classifier = SklearnClassifier(MultinomialNB()).train(trainfeats)
print('Multinomial NB accuracy:', nltk.classify.util.accuracy(classifier, testfeats))

Bernoulli NB accuracy: 0.812
Multinomial NB accuracy: 0.83


In [ ]:
from sklearn.svm import LinearSVC

classifier = SklearnClassifier(LinearSVC()).train(trainfeats)
print('Linear SVM accuracy:', nltk.classify.util.accuracy(classifier, testfeats))

Linear SVM accuracy: 0.864


In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = SklearnClassifier(LogisticRegression(max_iter=2000)).train(trainfeats)
print('Logistic Regression accuracy:', nltk.classify.util.accuracy(classifier, testfeats))

Logistic Regression accuracy: 0.896


NLTK offers no other easy metric computation methods besides accuracy, so if you would like to compute precision, recall, etc., you must unpack the previous list of (word_dict, label) pairs to make predictions.

In [ ]:
testfeats_nolabel, y_val = zip(*testfeats)
y_pred = classifier.classify_many(testfeats_nolabel)

In [ ]:
def compute_metrics(y_val, y_pred, pos_class):
    """
    Computes accuracy, precision, recall, and f1 for
    model predictions and specified positive class.

    Args:
        y_val (list): validation set predictions
        y_pred (list): true labels of validation set
        pos_class (str): determines which class precision/recall are evaluated on

    Returns:
        (list): list containing each of the above metrics
    """
    tp, tn, fp, fn = 0, 0, 0, 0

    for true, pred in zip(y_val, y_pred):
      if pred == pos_class:
        if true == pos_class:
          tp += 1
        else:
          fp += 1

      elif pred != pos_class:
        if true != pos_class:
          tn += 1
        else:
          fn += 1

    accuracy = (tp + tn+1) / (tp + tn + fp + fn+1)
    precision = (tp+1) / (tp + fp+1)
    recall = (tp+1) / (tp + fn+1)
    f1 = 2*(precision*recall) / (precision+recall)

    return [accuracy, precision, recall, f1]

In [ ]:
compute_metrics(y_val, y_pred, "pos")

[0.8962075848303394, 0.927038626609442, 0.8605577689243028, 0.8925619834710744]

## K-Fold

K-Fold is a more robust way to determine the efficacy of a model, because it allows us to compute aggregated measures across multiple subsets of the train/validation sets.

In [ ]:
import copy
import random

def perform_kfold(k, model, data, pos_class):
  """
  Performs K-fold cross-validation by dividing the dataset into k (relatively) equal
  size folds for train and validation sets.

  Metrics are averaged across the folds.

  Args:
      k (int): number of folds
      model: NLTK-type classifier
      data (list): list of tuple pairs containing a word dictionary, and the class label
      pos_class (str): which class to evaluate as the positive class

  Returns:
      (dict): dict object containing the averaged metrics
  """
  metrics = []

  # Good idea to shuffle data here, as we have just previously performed a superficial
  # juggling of the positive and negative classes
  shuffled = random.shuffle(copy.deepcopy(data))

  n_samples = len(data)
  start = 0
  end = 0

  for i in range(k):
    if i+1 > n_samples % k:
      end += n_samples // k
    else:
      # First n_samples % k folds get an additional element to minimize unevenness
      end += n_samples // k + 1

    train = data[:start] + data[end:]

    # This zip(*) unpacks the tuple with the word dictionary and labels
    val, y_val = zip(*data[start:end])

    # Train the model and make predictions for metric computation
    model.train(train)
    y_pred = model.classify_many(val)

    metrics.append(compute_metrics(y_val, y_pred, pos_class))

    start = end

  # Compute averages of metrics
  metrics = np.round(np.average(metrics, axis=0), 3)
  metrics = {"accuracy": metrics[0],
             "precision": metrics[1],
             "recall": metrics[2],
             "f1": metrics[3]}

  return metrics

In [ ]:
perform_kfold(
    k=4,
    model=SklearnClassifier(LogisticRegression(max_iter=2000)),
    data= trainfeats + testfeats,
    pos_class = "pos"
)

{'accuracy': 0.804, 'precision': 0.708, 'recall': 0.86, 'f1': 0.656}

#Feature Selection

## Frequency-based

In [ ]:
from nltk import FreqDist, word_tokenize

all_words = word_tokenize(movie_reviews.raw()) # Tokenizing every term
freq_dist = FreqDist(all_words) # Finding how many times each term occurs overall

print("Total number of terms: %d" % len(freq_dist))
print(freq_dist.most_common(25))

Total number of terms: 46462
[(',', 77717), ('the', 76276), ('.', 65876), ('a', 37995), ('and', 35404), ('of', 33972), ('to', 31772), ('is', 26054), ('in', 21611), ("'s", 18128), ('``', 17625), ('it', 16059), ('that', 15912), (')', 11781), ('(', 11664), ('as', 11349), ('with', 10782), ('for', 9918), ('this', 9573), ('his', 9569), ('film', 9443), ('i', 8850), ('he', 8840), ('but', 8604), ('on', 7249)]


Based on this small subset of terms (25), the most frequent terms appear to mostly be either stop words or punctuation. So, it may be a good idea to remove both the frequent, and infrequent terms.

See if you can work with the above frequency distribution to remove words that say, appear in less than 10 documents.

## Mutual Information

Another alternative to just selecting words by frequency is to determine how much their presence/absence correlates to a paticular class. This can be done through computing the mutual information gain with respect to each word.

In [ ]:
# Collecting the positive and negative associated instances
pos = [word_feats(movie_reviews.words(f)) for f in posids]
neg = [word_feats(movie_reviews.words(f)) for f in negids]

# Convert positive and negative representations into dataframes
pos_df = pd.DataFrame(pos)
pos_df['label'] = 'pos' # Adding labels just to split them
neg_df = pd.DataFrame(neg)
neg_df['label'] = 'neg'

# Concat so that pos and neg can be split and still have equal columns
full = pd.concat((pos_df, neg_df), axis=0).fillna(0)

# Extract the positive and negative documents again
pos_df = full.drop('label', axis=1).loc[full['label'] == 'pos', :]
neg_df = full.drop('label', axis=1).loc[full['label'] == 'neg', :]

In [ ]:
# Will serve as the total word pool
word_dict = dict.fromkeys(full.columns.drop('label'))

# Constants
pos_size = len(pos_df)
neg_size = len(neg_df)
N = pos_size + neg_size

In [ ]:
# Reference: https://nlp.stanford.edu/IR-book/html/htmledition/mutual-information-1.html
for word in word_dict.keys():
    n11 = pos_df[word].sum() # Number of class 1 documents that contain current word
    n10 = neg_df[word].sum() # Number of class 2 documents ...
    n01 = pos_size - n11 # Number of class 1 documents that do not contain current word
    n00 = neg_size - n10 # Number of class 2 documents ...

    # Breaking up equation for better readability
    a = (n11/N)*np.log2(
        (N*n11+1)/((n11+n10)*(n11+n01)+1)
        )
    b = (n01/N)*np.log2(
        (N*n01+1)/((n01+n00)*(n11+n01)+1)
        )
    c = (n10/N)*np.log2(
        (N*n10+1)/((n11+n10)*(n10+n00)+1)
        )
    d = (n00/N)*np.log2(
        (N*n00+1)/((n01+n00)*(n10+n00)+1)
        )

    # Add all eq. terms
    I = a + b + c + d

    # If a value is so small that a runtime error occurs just set resulting
    # information score to 0
    if I != I:
      word_dict[word] = 0
    else:
      word_dict[word] = I

In [ ]:
# Sort the mutual information scores
sorted_I = pd.Series(word_dict).sort_values(ascending=False)

print(sorted_I.head(25)) # Display the 25 highest ranked values

# Select M = 7000 terms for the subset
selected_subset = sorted_I[:7000].to_dict() # Convert to dict for O(1) matching

bad             0.050170
worst           0.041427
stupid          0.029463
boring          0.028749
ridiculous      0.024619
waste           0.024619
awful           0.024291
wasted          0.022762
outstanding     0.021744
mess            0.020948
lame            0.020029
perfect         0.018938
life            0.018661
supposed        0.018642
wonderfully     0.018258
memorable       0.017486
dull            0.016900
poorly          0.016468
excellent       0.015788
perfectly       0.015242
both            0.015229
script          0.015213
plot            0.014770
subtle          0.014769
performances    0.014655
dtype: float64


In [ ]:
def word_feats_MI(words: list, subset: dict) -> dict:
  """
  Adaptation of word feats to select top N words by Mutual Information (MI)

  Args:
    words (list): list of words/tokens occuring in document

  Returns:
    (dict): dictionary with words as keys (values just indicate the word occurs)
  """
  selected_words = {}
  for word in words:
    try: # Check if word is in the subset selected with MI
      subset[word]
      selected_words[word] = True
    except KeyError as e: # Word is not in the subset, just ignore it
      pass

  return selected_words

# Convert sentences into feature vectors
negfeats_mi = [(word_feats_MI(movie_reviews.words(fileids=[f]), selected_subset), 'neg') for f in negids]
posfeats_mi = [(word_feats_MI(movie_reviews.words(fileids=[f]), selected_subset), 'pos') for f in posids]

# TF-IDF

In [ ]:
# Determining the frequency of terms in each document
pos = [FreqDist(movie_reviews.words(f)) for f in posids]
neg = [FreqDist(movie_reviews.words(f)) for f in negids]

# Adding the labels for each document class
pos_df = pd.DataFrame(pos)
pos_df['label'] = 'pos'
neg_df = pd.DataFrame(neg)
neg_df['label'] = 'neg'

# Concat the pos and neg documents together and fill nan values
# (makes it easier to compute the tf and idf)
full = pd.concat((pos_df, neg_df), axis=0).fillna(0)

# Compute term frequencies
# Divide each term (column) for each document by the sum of each row
tf_matrix = full.drop('label', axis=1).div(full.drop('label', axis=1).sum(axis=1), axis=0)

# Compute inverse document frequencies
# Take the sum of each column and then divide by the number of documents
idf_matrix = full.drop('label', axis=1).sum(axis=0).div(len(full))

# Finding TF-IDF weight
tf_idf = tf_matrix*idf_matrix

# Seperate the positive and negative instances again, and replace 0 with nans
# (makes it easier to manipulate the dataframe back into the NLTK format)
pos_tf_idf = tf_idf.loc[full['label']=='pos', :].replace(0, np.nan)
neg_tf_idf = tf_idf.loc[full['label']=='neg', :].replace(0, np.nan)

# Convert the positive and negative TF-IDF weights back into the NLTK format
posfeats_tf_idf = []
for _, row in pos_tf_idf.iterrows():
    posfeats_tf_idf.append((row.dropna().to_dict(), 'pos')) # <- row.dropna is why 0's are replaced above

negfeats_tf_idf = []
for _, row in neg_tf_idf.iterrows():
    negfeats_tf_idf.append((row.dropna().to_dict(), 'neg'))

print(posfeats_tf_idf[0][0])

{'films': 0.0008909512761020882, 'adapted': 2.668213457076566e-05, 'from': 0.02319721577726218, 'comic': 0.0011281902552204176, 'books': 4.5243619489559165e-05, 'have': 0.005685614849187935, 'had': 0.0026902552204176333, 'plenty': 7.77262180974478e-05, 'of': 0.2771009280742459, 'success': 0.00012529002320185614, ',': 1.938417053364269, 'whether': 0.00012587006960556845, 'they': 0.005597447795823666, "'": 0.44351798143851506, 're': 0.0006566125290023201, 'about': 0.00817401392111369, 'superheroes': 6.960556844547564e-06, '(': 0.12178190255220417, 'batman': 0.00011774941995359629, 'superman': 1.508120649651972e-05, 'spawn': 4.756380510440835e-05, ')': 0.12300348027842227, 'or': 0.005477958236658933, 'geared': 8.120649651972158e-06, 'toward': 5.80046403712297e-05, 'kids': 0.0001902552204176334, 'casper': 1.334106728538283e-05, 'the': 2.041957076566125, 'arthouse': 1.740139211136891e-06, 'crowd': 4.6403712296983755e-05, 'ghost': 4.466357308584687e-05, 'world': 0.0012030162412993038, 'but':

# Removing Stop Words and Punctuation


In [ ]:
from nltk.corpus import stopwords
stopwords = stopwords.words('english')
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

Based on examinination of the corpus, residual portions of a word with punctuation like t, s, re will be removed alongside the portions preceding the puncutation. However, there will still be punctuation marks leftover so let's try and remove those as well.

In [ ]:
def word_feats_stop(words: list, stopwords: list, marks: list) -> dict:
  """
  Modification of word_feats function to remove stop words and punctuation

  Args:
    words (list): list of words/tokens occuring in document
    stopwords (list): list of stopwords to avoid including
    marks (list): list of marks to avoid including

  Returns:
    (dict): dictionary with words as keys (values just indicate the word occurs)
  """
  lst = []
  for word in words:
    if (word not in stopwords) and (word not in marks):
      lst.append((word, True))

  return dict(lst)

marks = ', " . ? ! & ; : ! { } [ ] ( ) \\ - _ `` /'.split(' ') # List of punctuation marks

negfeats_stop = [(word_feats_stop(movie_reviews.words(fileids=[f]), stopwords, marks), 'neg') for f in negids]
posfeats_stop = [(word_feats_stop(movie_reviews.words(fileids=[f]), stopwords, marks), 'pos') for f in posids]

print('Original dimensions: %g' % len(negfeats[0][0]))
print('Reduced dimensions: %g' % len(negfeats_stop[0][0]))


Original dimensions: 354
Reduced dimensions: 257


# Stemming

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def word_feats_stem(words: list, stemmer: PorterStemmer) -> dict:
  """
  Modification of word_feats function to stem words

  Args:
    words (list): list of words/tokens occuring in document
    stemmer (PorterStemmer): stemmer (can be any object implementing stem())

  Returns:
    (dict): dictionary with words as keys (values just indicate the word occurs)
  """
  return dict([(stemmer.stem(word), True) for word in words])

negfeats_stem = [(word_feats_stem(movie_reviews.words(fileids=[f]), stemmer), 'neg') for f in negids]
posfeats_stem = [(word_feats_stem(movie_reviews.words(fileids=[f]), stemmer), 'pos') for f in posids]

counter = 0
for _, (k1, k2) in enumerate(zip(negfeats[0][0], negfeats_stem[0][0])):
  if k1 != k2:
    print(k1, '->', k2)
    counter += 1
  elif counter == 10:
    break

couples -> coupl
party -> parti
accident -> accid
guys -> guy
dies -> die
his -> hi
continues -> continu
has -> ha
nightmares -> nightmar
movie -> movi


Adjusting this to use lemmatization should take little effort, try it out to see what the difference is between the two!

https://www.nltk.org/api/nltk.stem.wordnet.html

# Unigrams + Bigrams

In [ ]:
from nltk.util import ngrams

def word_feats_bigram(words):
    """
    Modification of word_feats function to include bigrams

    Args:
      words (list): list of words/tokens occuring in document

    Returns:
      (dict): dictionary with words as keys (values just indicate the word occurs)
    """

    lst = [word for word in words]
    bg = lst + list(ngrams(lst, 2)) # Append bigram terms to unigram terms
    feats = dict([(word, True) for word in bg])

    return feats

negfeats_bg = [(word_feats_bigram(movie_reviews.words(fileids=[f])), 'neg') for f in negids]
posfeats_bg = [(word_feats_bigram(movie_reviews.words(fileids=[f])), 'pos') for f in posids]

Here all we are doing is adding word co-occurences into the word dictionary for each document. Feel free to try this out with trigrams, 4-grams, etc. (note that the time to train and perform K-fold will take significantly longer)

In [ ]:
list(negfeats_bg[0][0].keys())[-10:]

[('-', 'memento'),
 ('memento', '('),
 ('the', 'others'),
 ('others', '('),
 ('-', 'stir'),
 ('stir', 'of'),
 ('of', 'echoes'),
 ('echoes', '('),
 ('(', '8'),
 ('8', '/')]